# Notebook 16 - Dual-View Adapter Inference Ablation

Runs the same adapter on whole image and router/SAM ROI crop, but only lets the ROI prediction compete when router crop/part match the adapter target.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _ensure_aads_repo_on_path():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, CLONE_TARGET, Path('/content/bitirmeprojesi'), Path('/content/bitirme projesi')]
    for candidate in candidates:
        if (candidate / 'src').is_dir() and (candidate / 'scripts').is_dir():
            repo_root = candidate.resolve()
            os.chdir(repo_root)
            try:
                subprocess.run(['git', 'pull', '--ff-only'], check=True)
            except Exception as exc:
                print(f'[BOOT] git pull skipped/failed: {exc}', flush=True)
            if str(repo_root) not in sys.path:
                sys.path.insert(0, str(repo_root))
            return repo_root
    if not CLONE_TARGET.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
    if str(CLONE_TARGET) not in sys.path:
        sys.path.insert(0, str(CLONE_TARGET))
    os.chdir(CLONE_TARGET)
    return CLONE_TARGET

ROOT = _ensure_aads_repo_on_path()
from scripts.colab_notebook_bootstrap_helpers import setup_notebook_environment
ROOT = setup_notebook_environment(install_requirements=True, print_tokens=True)
print(f'[BOOT] ROOT={ROOT}')


In [ ]:
from pathlib import Path
import importlib
import scripts.colab_roi_ablation as roi_ablation

roi_ablation = importlib.reload(roi_ablation)
commit_and_push_ablation_results = roi_ablation.commit_and_push_ablation_results
discover_dual_view_targets = roi_ablation.discover_dual_view_targets
run_dual_view_inference_targets = roi_ablation.run_dual_view_inference_targets

ABLATION_NAME = 'dual_view_inference'
DATASET_ROOT = ROOT / 'data' / 'prepared_runtime_datasets'
RUNS_ROOT = ROOT / 'runs'
TARGETS = []
INCLUDE_DATASET_KEYS = []
MAX_TARGETS = None
OUTPUT_DIR = ROOT / 'docs' / 'ablation_results' / ABLATION_NAME
CONFIG_ENV = 'colab'
DEVICE = 'cuda'
ROI_CONFIDENCE_MARGIN = 0.10
FULL_CONFIDENCE_REVIEW_THRESHOLD = 0.70
REQUIRE_SEMANTIC_ROI_MATCH = True
TARGET_ROI_BACKEND = 'router_then_grounding_dino'
GROUNDING_DINO_MODEL_ID = 'IDEA-Research/grounding-dino-tiny'
GROUNDING_DINO_BOX_THRESHOLD = 0.15
GROUNDING_DINO_TEXT_THRESHOLD = 0.10
GROUNDING_DINO_MAX_CANDIDATES = 5

print(f'[CONFIG] ablation={ABLATION_NAME} dataset_root={DATASET_ROOT} output={OUTPUT_DIR}')


In [ ]:
if not Path(DATASET_ROOT).is_dir():
    raise FileNotFoundError(f'DATASET_ROOT not found: {DATASET_ROOT}')
if not Path(RUNS_ROOT).is_dir():
    raise FileNotFoundError(f'RUNS_ROOT not found: {RUNS_ROOT}')

targets = TARGETS or discover_dual_view_targets(
    ROOT,
    dataset_root=DATASET_ROOT.relative_to(ROOT),
    runs_root=RUNS_ROOT.relative_to(ROOT),
    include_dataset_keys=INCLUDE_DATASET_KEYS or None,
)
if MAX_TARGETS is not None:
    targets = targets[:int(MAX_TARGETS)]
if not targets:
    raise RuntimeError('No dual-view targets found. Prepare matching runtime datasets and adapter exports, or fill TARGETS.')
print('[TARGETS]', [target.get('target_id') or target.get('dataset_key') for target in targets])

report = run_dual_view_inference_targets(
    repo_root=ROOT,
    targets=targets,
    output_dir=OUTPUT_DIR,
    config_env=CONFIG_ENV,
    device=DEVICE,
    roi_confidence_margin=ROI_CONFIDENCE_MARGIN,
    full_confidence_review_threshold=FULL_CONFIDENCE_REVIEW_THRESHOLD,
    require_semantic_roi_match=REQUIRE_SEMANTIC_ROI_MATCH,
    target_roi_backend=TARGET_ROI_BACKEND,
    grounding_dino_model_id=GROUNDING_DINO_MODEL_ID,
    grounding_dino_box_threshold=GROUNDING_DINO_BOX_THRESHOLD,
    grounding_dino_text_threshold=GROUNDING_DINO_TEXT_THRESHOLD,
    grounding_dino_max_candidates=GROUNDING_DINO_MAX_CANDIDATES,
)
git_result = commit_and_push_ablation_results(
    OUTPUT_DIR,
    repo_root=ROOT,
    message='Add dual-view ROI inference ablation results',
)
{'summary': report['summary'], 'git': git_result}
